# CuML vs scikit-learn: Notebook de demostración

Cuaderno de apoyo a la documentación. Incluye celdas ejecutables y algunas comentadas para que puedas mostrar el flujo en vivo.

## 0. Entorno y versiones

In [1]:
import importlib
import platform
import time

print("Python:", platform.python_version())
for pkg in ["cuml", "cudf", "cupy", "sklearn", "xgboost"]:
    spec = importlib.util.find_spec(pkg)
    if spec:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "(sin __version__)")
        print(f"{pkg}: {ver}")
    else:
        print(f"{pkg}: no instalado")

Python: 3.12.12
cuml: 25.10.00
cudf: 25.10.00
cupy: 13.6.0
sklearn: 1.6.1
xgboost: 3.1.2


In [3]:
import cuml
import cudf
import cupy as cp
print("cuML", cuml.__version__)
print("cuDF", cudf.__version__)
print(f"GPU disponible? {cp.cuda.runtime.getDeviceCount() }")

cuML 25.10.00
cuDF 25.10.00
GPU disponible? 1


## 1. Carga de datos de ejemplo
Breast Cancer para clasificación y California Housing para regresión (ambos vienen en sklearn).

In [4]:
from sklearn.datasets import load_breast_cancer, fetch_california_housing

X_cls, y_cls = load_breast_cancer(return_X_y=True)
X_reg, y_reg = fetch_california_housing(return_X_y=True)
X_cls.shape, X_reg.shape

((569, 30), (20640, 8))

## 2. Regresión logística: CPU vs GPU (si hay CuML)

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Xtr, Xte, ytr, yte = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# CPU
log_cpu = LogisticRegression(max_iter=5000)
start = time.time()
log_cpu.fit(Xtr, ytr)
cpu_time = time.time() - start
cpu_acc = accuracy_score(yte, log_cpu.predict(Xte))
print(f"CPU accuracy={cpu_acc:.3f} tiempo={cpu_time:.3f}s")

try:
    import cupy as cp
    from cuml.linear_model import LogisticRegression as cuLogReg
    Xtr_cp, Xte_cp = cp.asarray(Xtr), cp.asarray(Xte)
    ytr_cp, yte_cp = cp.asarray(ytr), cp.asarray(yte)
    start = time.time()
    log_gpu = cuLogReg(max_iter=5000)
    log_gpu.fit(Xtr_cp, ytr_cp)
    gpu_time = time.time() - start
    gpu_acc = float(accuracy_score(cp.asnumpy(yte_cp), cp.asnumpy(log_gpu.predict(Xte_cp))))
    print(f"GPU accuracy={gpu_acc:.3f} tiempo={gpu_time:.3f}s")
except Exception as e:
    print("GPU no disponible o fallo cuML:", e)

CPU accuracy=0.965 tiempo=0.456s
GPU accuracy=0.965 tiempo=2.037s


## 3. Random Forest (clasificación)

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf_cpu = RandomForestClassifier(n_estimators=200,  random_state=42)
rf_cpu.fit(Xtr, ytr)
print("CPU accuracy RF:", accuracy_score(yte, rf_cpu.predict(Xte)))

try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRF
    Xtr_gdf = cudf.DataFrame(Xtr)
    ytr_gdf = cudf.Series(ytr)
    Xte_gdf = cudf.DataFrame(Xte)
    rf_gpu = cuRF(n_estimators=300,  random_state=42)
    rf_gpu.fit(Xtr_gdf, ytr_gdf)
    preds = rf_gpu.predict(Xte_gdf)
    acc_gpu = float(accuracy_score(yte, preds.to_numpy()))
    print("GPU accuracy RF:", acc_gpu)
except Exception as e:
    print("RF GPU no disponible:", e)

CPU accuracy RF: 0.956140350877193
GPU accuracy RF: 0.9649122807017544


In [20]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from cuml.neighbors import KNeighborsClassifier
from cuml.metrics import accuracy_score as cuml_accuracy_score
import cudf

X_np, y_np = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
knn_cpu = KNeighborsClassifier(n_neighbors=5)
start = time.time();knn_cpu.fit(X_tr, y_tr);end_time = time.time()
print(f"KNN CPU training time: {end_time - start:.3f} seconds")
pred_cpu = knn_cpu.predict(X_te)
acc_knn_cpu = accuracy_score(y_te, pred_cpu)
print(f"KNN CPU accuracy: {acc_knn_cpu:.3f}")

X_tr_gdf = cudf.DataFrame(X_tr)
X_te_gdf = cudf.DataFrame(X_te)
y_tr_gdf = cudf.Series(y_tr)

knn_gpu = KNeighborsClassifier(n_neighbors=5)
start = time.time();knn_gpu.fit(X_tr_gdf, y_tr_gdf);end_time = time.time()
print(f"KNN GPU training time: {end_time - start:.3f} seconds")
pred = knn_gpu.predict(X_te_gdf)
acc_knn_gpu = float(cuml_accuracy_score(y_te, pred.to_numpy()))
print(f"KNN GPU accuracy: {acc_knn_gpu:.3f}")

KNN CPU training time: 0.003 seconds
KNN CPU accuracy: 0.722
KNN GPU training time: 0.004 seconds
KNN GPU accuracy: 0.722


## 4. K-Means (no supervisado)

In [21]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans as SKKMeans

X_blobs, _ = make_blobs(n_samples=200000, centers=10, n_features=20, random_state=0)
km_cpu = SKKMeans(n_clusters=10, n_init=10)
start = time.time(); km_cpu.fit(X_blobs); cpu_t = time.time()-start
print(f"KMeans CPU tiempo={cpu_t:.3f}s")

try:
    import cudf
    from cuml.cluster import KMeans as cuKMeans
    X_gdf = cudf.DataFrame(X_blobs.astype('float32'))
    start = time.time(); km_gpu = cuKMeans(n_clusters=10); km_gpu.fit(X_gdf); gpu_t = time.time()-start
    print(f"KMeans GPU tiempo={gpu_t:.3f}s")
except Exception as e:
    print("KMeans GPU no disponible:", e)

KMeans CPU tiempo=1.609s
KMeans GPU tiempo=0.029s


## 5. Pipeline simple en GPU

In [6]:
try:
    import cudf
    from cuml.preprocessing import StandardScaler
    from cuml.linear_model import LogisticRegression as cuLogReg
    from cuml import Pipeline

    df = cudf.DataFrame(X_cls)
    y = cudf.Series(y_cls)
    pipe_gpu = Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', cuLogReg(max_iter=1000)),
    ])
    pipe_gpu.fit(df, y)
    print("Pipeline GPU entrenado correctamente")
except Exception as e:
    print("Pipeline GPU no disponible:", e)

Pipeline GPU no disponible: cannot import name 'Pipeline' from 'cuml' (/usr/local/lib/python3.12/dist-packages/cuml/__init__.py)


## 6. Búsqueda de hiperparámetros (loop reducido en GPU)

In [ ]:
try:
    import cudf
    from cuml.ensemble import RandomForestClassifier
    from cuml.metrics import accuracy_score
    from sklearn.model_selection import train_test_split

    Xtr, Xva, ytr, yva = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)
    Xtr_gdf, Xva_gdf = cudf.DataFrame(Xtr), cudf.DataFrame(Xva)
    ytr_gdf, yva_gdf = cudf.Series(ytr), cudf.Series(yva)

    param_grid = [
        {'n_estimators': 200, 'max_depth': 10},
        {'n_estimators': 400, 'max_depth': 12},
    ]
    best_score, best_params = -1, None
    for params in param_grid:
        model = RandomForestClassifier(random_state=42, **params)
        model.fit(Xtr_gdf, ytr_gdf)
        preds = model.predict(Xva_gdf)
        score = accuracy_score(yva_gdf, preds)
        if score > best_score:
            best_score, best_params = score, params
    print('Mejor combinación:', best_params, 'accuracy:', float(best_score))
except Exception as e:
    print("No se pudo ejecutar búsqueda GPU:", e)

## 7. UMAP en GPU (reducción de dimensión)

In [ ]:
try:
    import cudf
    from cuml.manifold import UMAP
    from sklearn.datasets import load_digits

    digits = load_digits()
    X_gdf = cudf.DataFrame(digits.data.astype('float32'))

    umap = UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=0)
    embedding = umap.fit_transform(X_gdf)
    print(embedding.head())
except Exception as e:
    print("UMAP GPU no disponible:", e)

## 8. XGBoost con backend GPU

In [ ]:
try:
    import xgboost as xgb
    import cudf
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    df = cudf.DataFrame(X_cls)
    df['target'] = y_cls
    X_tr, X_te, y_tr, y_te = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42, stratify=df['target'])

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dtest = xgb.DMatrix(X_te, label=y_te)
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'gpu_hist',
        'max_depth': 6,
        'learning_rate': 0.1,
        'eval_metric': 'logloss',n_estimators=200
    }
    bst = xgb.train(params, dtrain, num_boost_round=200)
    preds = (bst.predict(dtest) > 0.5).astype(int)
    print("XGBoost GPU accuracy:", accuracy_score(y_te.to_pandas(), preds))
except Exception as e:
    print("XGBoost GPU no disponible:", e)

## 9. Notas y próximos pasos
- Añade/activa celdas comentadas si quieres probar otros modelos (UMAP ya incluido, XGBoost GPU incluido).
- Si no hay GPU/cuML instalados, las celdas de GPU fallarán mostrando el mensaje de error.